In [1]:
print("Hello World")

Hello World


In [2]:
import tensorflow as tf

In [3]:
from tensorflow.keras.applications import DenseNet121
model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

In [4]:
model.trainable = False

In [5]:
from tensorflow.keras.layers import GlobalAveragePooling2D,Dense


In [6]:
x = GlobalAveragePooling2D()(model.output)
output = Dense(4,activation="softmax")(x)

In [7]:
MODEL = tf.keras.Model(inputs=model.input,outputs=output)

In [8]:
MODEL.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=['accuracy']
)

In [9]:
from tensorflow.keras.applications.densenet import preprocess_input

image_size=(224,224)
batch_size=32

In [10]:
train = tf.keras.utils.image_dataset_from_directory(
    './data/stage1_balanced/train',
    image_size=image_size,
    batch_size=batch_size,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

Found 4849 files belonging to 4 classes.


In [11]:
valid = tf.keras.utils.image_dataset_from_directory(
    './data/stage1_balanced/val',
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False,
    label_mode="categorical"
)

Found 1614 files belonging to 4 classes.


In [12]:
test = tf.keras.utils.image_dataset_from_directory(
    './data/stage1_balanced/test',
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False,
    label_mode="categorical"
)

Found 1621 files belonging to 4 classes.


In [13]:
print("Classes:", train.class_names)


Classes: ['Corona Virus Disease', 'Normal', 'Pneumonia', 'Tuberculosis']


In [14]:
AUTOTUNE = tf.data.AUTOTUNE
def preprocess(image,label):
    images = tf.cast(image, tf.float32)
    images = preprocess_input(images)
    return images, label

In [27]:
train = train.map(preprocess,num_parallel_calls=AUTOTUNE)
valid = valid.map(preprocess,num_parallel_calls=AUTOTUNE)
test = test.map(preprocess,num_parallel_calls=AUTOTUNE)

In [28]:
train = train.prefetch(AUTOTUNE)
valid = valid.prefetch(AUTOTUNE)
test = test.prefetch(AUTOTUNE)

In [21]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-7
    )
]

In [22]:
history = MODEL.fit(
    train,
    validation_data=valid,
    callbacks=callbacks,
    verbose=1,
    epochs=100
)

Epoch 1/100


c:\Users\smitp\.conda\envs\Medical\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


152/152 ━━━━━━━━━━━━━━━━━━━━ 143s 837ms/step - accuracy: 0.5682 - loss: 1.2077 - val_accuracy: 0.6351 - val_loss: 1.1266 - learning_rate: 1.0000e-04
Epoch 2/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 104s 682ms/step - accuracy: 0.6626 - loss: 1.0672 - val_accuracy: 0.6673 - val_loss: 1.0158 - learning_rate: 1.0000e-04
Epoch 3/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 104s 685ms/step - accuracy: 0.6857 - loss: 0.9768 - val_accuracy: 0.6834 - val_loss: 0.9418 - learning_rate: 1.0000e-04
Epoch 4/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 105s 688ms/step - accuracy: 0.6991 - loss: 0.9136 - val_accuracy: 0.6927 - val_loss: 0.8891 - learning_rate: 1.0000e-04
Epoch 5/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 105s 691ms/step - accuracy: 0.7113 - loss: 0.8670 - val_accuracy: 0.7057 - val_loss: 0.8491 - learning_rate: 1.0000e-04
Epoch 6/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 103s 676ms/step - accuracy: 0.7214 - loss: 0.8304 - val_accuracy: 0.7094 - val_loss: 0.8172 - learning_rate: 1.0000e-04
Epoch 7/100
152/152 ━━━━━━━━━━━━━━━━━━━━ 103s 

In [29]:
import numpy as np

y_true = []
y_pred = []

for images, labels in test:
    predictions = MODEL.predict(images, verbose=0)

    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("y_true:", y_true.shape)
print("y_pred:", y_pred.shape)

y_true: (1621,)
y_pred: (1621,)


In [26]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nMacro F1:",
    f1_score(y_true, y_pred, average="macro", zero_division=0))

print("Weighted F1:",
    f1_score(y_true, y_pred, average="weighted", zero_division=0))

Accuracy: 0.2998149290561382

Classification Report:
              precision    recall  f1-score   support

           0     0.8000    0.0491    0.0926       407
           1     0.0156    0.0025    0.0043       404
           2     0.7816    0.1692    0.2781       402
           3     0.2747    0.9730    0.4285       408

    accuracy                         0.2998      1621
   macro avg     0.4680    0.2985    0.2009      1621
weighted avg     0.4677    0.2998    0.2011      1621


Confusion Matrix:
[[ 20  58   2 327]
 [  0   1  13 390]
 [  1   2  68 331]
 [  4   3   4 397]]

Macro F1: 0.20086975994656792
Weighted F1: 0.20113574951676158


In [30]:
print("MODEL input:", MODEL.input_shape)
print("MODEL output:", MODEL.output_shape)
print("Classes:", train.class_names)
print("Valid classes:", valid.class_names)
print("Test classes:", test.class_names)

MODEL input: (None, 224, 224, 3)
MODEL output: (None, 4)


AttributeError: '_PrefetchDataset' object has no attribute 'class_names'

In [31]:
from pathlib import Path

train_dir = Path(r"C:\AI\Medical\Liver\data\stage1_balanced\train")
valid_dir = Path(r"C:\AI\Medical\Liver\data\stage1_balanced\val")
test_dir  = Path(r"C:\AI\Medical\Liver\data\stage1_balanced\test")

print("Train classes:", sorted([x.name for x in train_dir.iterdir() if x.is_dir()]))
print("Valid classes:", sorted([x.name for x in valid_dir.iterdir() if x.is_dir()]))
print("Test classes:", sorted([x.name for x in test_dir.iterdir() if x.is_dir()]))

Train classes: ['Corona Virus Disease', 'Normal', 'Pneumonia', 'Tuberculosis']
Valid classes: ['Corona Virus Disease', 'Normal', 'Pneumonia', 'Tuberculosis']
Test classes: ['Corona Virus Disease', 'Normal', 'Pneumonia', 'Tuberculosis']


In [32]:
print("MODEL input:", MODEL.input_shape)
print("MODEL output:", MODEL.output_shape)

MODEL input: (None, 224, 224, 3)
MODEL output: (None, 4)


In [33]:
import numpy as np

y_true_val = []
y_pred_val = []

for images, labels in valid:
    preds = MODEL.predict(images, verbose=0)

    y_true_val.extend(np.argmax(labels.numpy(), axis=1))
    y_pred_val.extend(np.argmax(preds, axis=1))

y_true_val = np.array(y_true_val)
y_pred_val = np.array(y_pred_val)

print("Validation accuracy:",
    np.mean(y_true_val == y_pred_val))

print("Validation prediction counts:")
print(np.bincount(y_pred_val, minlength=4))

Validation accuracy: 0.25154894671623296
Validation prediction counts:
[1614    0    0    0]
